# Titanic Survival Prediction 🚢

## Project Overview
In this project, we analyze the Titanic dataset and build a machine learning model to predict passenger survival.

Main steps:
- Exploratory Data Analysis (EDA)
- Handling missing values
- Categorical and numerical feature analysis
- Feature engineering
- Building a Logistic Regression pipeline
- Model evaluation and error analysis
- Interpreting model coefficients


In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np

df = sns.load_dataset("titanic")

df.head()


## 1. Initial Data Exploration

In [ ]:
df.info()

df.isnull().sum()


## 2. Exploring Relationships With Survival

In [ ]:
# Compare numerical features between survivors and non-survivors

df.groupby("survived")[["age", "fare"]].mean()


In [ ]:
# Survival rate by gender

df.groupby("sex")["survived"].mean()


In [ ]:
# Boolean features can be converted to integers for correlation analysis

df[["survived", "age", "fare", "sibsp", "parch", "adult_male", "alone"]].corr()


In [ ]:
# Survival rate by passenger class

df.groupby("class", observed=True)["survived"].mean()


## 3. Categorical Feature Relationships

In [ ]:
# Relationship between gender and survival

pd.crosstab(df["sex"], df["survived"])


In [ ]:
# Measuring association between categorical variables using Cramer's V

from scipy.stats.contingency import association

table = pd.crosstab(df["sex"], df["survived"])
association(table.to_numpy(), method="cramer")


## 4. Feature Engineering

In [ ]:
# Combine gender and passenger class

df["sex_class"] = (
    df["sex"].astype(str) + "_" + df["class"].astype(str)
)

df.groupby("sex_class", observed=True)["survived"].agg(
    ["count", "mean"]
).sort_values("mean", ascending=False)


In [ ]:
# Create family size feature

df["family_size"] = df["sibsp"] + df["parch"] + 1

df[["sibsp", "parch", "family_size"]].head()


In [ ]:
# Convert family size into meaningful groups

df["family_group"] = pd.cut(
    df["family_size"],
    bins=[0, 1, 4, float("inf")],
    labels=["alone", "small", "large"]
)

df.groupby("family_group", observed=True)["survived"].agg(
    ["mean", "count"]
)


In [ ]:
# Interaction between family group and gender

df["sex_family"] = (
    df["family_group"].astype(str) + "_" + df["sex"].astype(str)
)

df.groupby(["sex", "family_group"], observed=True)["survived"].agg(
    ["mean", "count"]
)


## 5. Machine Learning Pipeline

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

numeric_features = ["age", "fare"]
categorical_base = ["sex", "class", "family_group"]
categorical_new = categorical_base + ["sex_family"]

X = df[categorical_new]
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:
def make_model(categorical_features):
    # Preprocess numerical and categorical features separately
    preprocess = ColumnTransformer([
        ("num", make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler()
        ), numeric_features),

        ("cat", make_pipeline(
            SimpleImputer(strategy="most_frequent"),
            OneHotEncoder(handle_unknown="ignore")
        ), categorical_features)
    ])

    # Combine preprocessing and Logistic Regression
    return make_pipeline(
        preprocess,
        LogisticRegression(max_iter=1000)
    )


model_a = make_model(categorical_base)
model_b = make_model(categorical_new)


In [ ]:
model_a.fit(X_train, y_train)
model_b.fit(X_train, y_train)

pred_a = model_a.predict(X_test)
pred_b = model_b.predict(X_test)

print("Model A:", accuracy_score(y_test, pred_a))
print("Model B:", accuracy_score(y_test, pred_b))


## 6. Model Evaluation

In [ ]:
print("=== Model A ===")
print(classification_report(y_test, pred_a))
print(confusion_matrix(y_test, pred_a))

print("=== Model B ===")
print(classification_report(y_test, pred_b))
print(confusion_matrix(y_test, pred_b))


## 7. Error Analysis

In [ ]:
errors = X_test.copy()

errors["actual"] = y_test
errors["prediction"] = pred_a

errors[errors["actual"] != errors["prediction"]].groupby(
    ["sex", "class", "actual", "prediction"]
).size().sort_values(ascending=False)


## 8. Logistic Regression Coefficient Analysis

In [ ]:
preprocessor = model_a.named_steps["columntransformer"]

feature_names = preprocessor.get_feature_names_out()

coefficients = model_a.named_steps["logisticregression"].coef_[0]

importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

importance.sort_values(
    "coefficient",
    ascending=False
)
